In [1]:
import pandas as pd
import numpy as np
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv('../.env')
token = os.getenv('ONEMAP_TOKEN')

print("Token loaded:", token is not None)

Token loaded: True


In [9]:
schools = pd.read_csv('../data/raw/schools.csv')
print(schools.shape)
print("\nmainlevel_code values:")
print(schools['mainlevel_code'].value_counts())

(337, 31)

mainlevel_code values:
mainlevel_code
PRIMARY                         179
SECONDARY (S1-S5)               117
SECONDARY (S1-S4)                16
JUNIOR COLLEGE                   10
MIXED LEVEL (S1-JC2)             10
MIXED LEVEL (P1-S4)               3
CENTRALISED INSTITUTE             1
MIXED LEVEL (S1-S5, JC1-JC2)      1
Name: count, dtype: int64


In [4]:
def geocode_postal(postal_code, token, retries=3):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": str(postal_code),
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    headers = {"Authorization": token}
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            data = response.json()
            if data.get('found', 0) > 0:
                result = data['results'][0]
                return float(result['LATITUDE']), float(result['LONGITUDE'])
            else:
                time.sleep(1)
        except Exception as e:
            time.sleep(1)
    return None, None

row = schools.iloc[0]
lat, lon = geocode_postal(row['postal_code'], token)
print(f"School: {row['school_name']}")
print(f"Postal: {row['postal_code']}")
print(f"Coords: {lat}, {lon}")

School: ADMIRALTY PRIMARY SCHOOL
Postal: 738907
Coords: 1.4426347903311, 103.800040119743


In [5]:
results = []

for idx, row in schools.iterrows():
    lat, lon = geocode_postal(row['postal_code'], token)
    results.append({
        'school_name': row['school_name'],
        'postal_code': row['postal_code'],
        'mainlevel_code': row['mainlevel_code'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

schools_geocoded = pd.DataFrame(results)
print(f"Done. Geocoded {len(schools_geocoded):,} schools.")
print(f"Failed: {schools_geocoded['latitude'].isna().sum()}")

schools_geocoded.to_csv('../data/processed/schools_geocoded.csv', index=False)
print("Saved to data/processed/schools_geocoded.csv")

Done. Geocoded 337 schools.
Failed: 0
Saved to data/processed/schools_geocoded.csv


In [7]:
import json

with open('../data/raw/hawker_centres.geojson', 'r') as f:
    hawker_data = json.load(f)

hawkers = []
for feature in hawker_data['features']:
    props = feature['properties']
    coords = feature['geometry']['coordinates']
    hawkers.append({
        'name': props['NAME'],
        'longitude': coords[0],
        'latitude': coords[1],
        'status': props['STATUS'],
        'est_completion_date': props['EST_ORIGINAL_COMPLETION_DATE']
    })

hawkers_df = pd.DataFrame(hawkers)
print(f"Total hawker centres: {len(hawkers_df)}")
print(hawkers_df['status'].value_counts())
hawkers_df.head()

Total hawker centres: 129
status
Existing                  103
Existing (new)             16
Under Construction          6
Existing (replacement)      3
Interim Centre              1
Name: count, dtype: int64


,name,longitude,latitude,status,est_completion_date
0,Commonwealth Crescent Market,103.800367,1.306900,Existing,1965
1,Tiong Bahru Market,103.832182,1.284786,Existing,NaN
2,Golden Mile Food Centre,103.863878,1.303142,Existing,1975
3,Yew Tee Hawker Centre,103.745922,1.397185,Under Construction,2027
4,Dunman Food Centre,103.901825,1.309418,Existing,1974


In [8]:
# Filter out centres under construction
hawkers_existing = hawkers_df[hawkers_df['status'] != 'Under Construction'].copy()
print(f"Hawker centres after filtering: {len(hawkers_existing)}")

hawkers_existing.to_csv('../data/processed/hawker_centres.csv', index=False)
print("Saved to data/processed/hawker_centres.csv")

Hawker centres after filtering: 123
Saved to data/processed/hawker_centres.csv


In [10]:
mrt = pd.read_csv('../data/raw/mrt_stations.csv')
print(f"MRT stations loaded: {len(mrt)}")
mrt.head()

MRT stations loaded: 143


,station_name,station_code,line,opening_date
0,Admiralty,NS10,North-South Line,1996-02-10
1,Aljunied,EW9,East-West Line,1989-11-04
2,Ang Mo Kio,NS16,North-South Line,1987-11-07
3,Bartley,CC12,Circle Line,2009-05-28
4,Bayfront,CE1/DT16,Circle Line Extension/Downtown Line,2012-01-14


In [11]:
def geocode_query(query, token, retries=3):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": query,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    headers = {"Authorization": token}
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            data = response.json()
            if data.get('found', 0) > 0:
                result = data['results'][0]
                return float(result['LATITUDE']), float(result['LONGITUDE'])
            else:
                time.sleep(1)
        except Exception as e:
            time.sleep(1)
    return None, None

# Test on one station
lat, lon = geocode_query("Admiralty MRT Station Singapore", token)
print(f"Admiralty MRT: {lat}, {lon}")

Admiralty MRT: 1.44058856161847, 103.800990519771


In [15]:
results = []

for idx, row in mrt.iterrows():
    # Include station code in query to disambiguate
    # Use first code only for multi-code stations
    primary_code = row['station_code'].split('/')[0]
    query = f"{row['station_name']} MRT Station ({primary_code})"
    lat, lon = geocode_query(query, token)
    results.append({
        'station_name': row['station_name'],
        'station_code': row['station_code'],
        'line': row['line'],
        'opening_date': row['opening_date'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

mrt_geocoded = pd.DataFrame(results)
print(f"Done. Geocoded {len(mrt_geocoded)} stations.")
print(f"Failed: {mrt_geocoded['latitude'].isna().sum()}")

# Check for duplicates
dupes = mrt_geocoded[mrt_geocoded.duplicated(subset=['latitude', 'longitude'], keep=False)]
print(f"Duplicate coordinates: {len(dupes)}")

Done. Geocoded 143 stations.
Failed: 0
Duplicate coordinates: 0


In [17]:
mrt_geocoded.to_csv('../data/processed/mrt_stations_geocoded.csv', index=False)
print("Saved to data/processed/mrt_stations_geocoded.csv")

Saved to data/processed/mrt_stations_geocoded.csv
